## MLO Fiber Spectrograph Exposure Time Calculator ##

In [116]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

In [118]:
# parameters (edit later) 
mag_zp       = 24.0    # photometric zero point
N_pix       = 6.5     # pixels in extraction aperture (6.5 px after 2x2 binning)
Read_noise   = 1.1     # e-/pixel
Dark_current = 0.0013  # e-/pixel/s  (QHY268 at -10 C, from datasheet plot)
Sky_noise    = 5.0     # sky background counts/s/pixel  (adjust once ZP is calibrated)

# QE at a few wavelengths for reference (from QHY268 / IMX571 datasheet):
#   450 nm -> 0.91    500 nm -> 0.89    550 nm -> 0.84
#   600 nm -> 0.75    650 nm -> 0.60    700 nm -> 0.46    750 nm -> 0.34
qe_wave = [400, 450, 500, 550, 600, 650, 700, 750, 800, 850]
qe_vals = [0.72, 0.91, 0.89, 0.84, 0.75, 0.60, 0.46, 0.34, 0.27, 0.22]
qe_curve = interp1d(qe_wave, qe_vals, kind='cubic', bounds_error=False, fill_value=0.0)

def get_qe(wave_nm):
    """takes wavelength in nm and returns QE value at that wavelength from the curve values above""" 
    return float(qe_curve(wave_nm))

In [120]:
# functions 
 
def counts(mag):
    """source counts/sec from magnitude"""
    return 10 ** ((mag_zp - mag) / 2.5)
 
 
def get_SNR(exp_time, mag, wave_nm=550.0, sky_noise=Sky_noise, dark=Dark_current,
            read_noise=Read_noise, n_pix=N_pix):
    """SNR given an exposure time and wavelength"""
    qe = get_qe(wave_nm)
    s  = counts(mag)
    numerator   = s * np.sqrt(qe * exp_time)
    denominator = np.sqrt(s + n_pix * (sky_noise + dark/qe + read_noise**2 / (qe * exp_time)))
    return numerator / denominator
 
 
def get_exptime(target_snr, mag, wave_nm=550.0, sky_noise=Sky_noise, dark=Dark_current,
                read_noise=Read_noise, n_pix=n_pix):
    """
    exposure time needed to reach a target SNR
    C =  grouping of all the noise terms that don't come from the star itself
    """
    qe = get_qe(wave_nm)
    s  = counts(mag)
    C  = s / n_pix + sky_noise + dark / qe
    t  = (
        C * n_pix * target_snr**2 * qe
        + np.sqrt((target_snr**2 * qe * n_pix * C)**2
                  + 4 * (s * qe)**2 * (read_noise**2 * n_pix * target_snr**2))
    ) / (2 * (s * qe)**2)
    return t




In [122]:
""" 
test with hard coded values to ensure functions are working properly: 

exp_time = 30
magnitude = 16
wavelength = 550 nm 
sky noise = read noise = 5
dark current = .01
n pix = (5pi)^2

"""
print(get_SNR(30.0, 16, wave_nm=550, sky_noise=5.0, dark=0.01, read_noise=5.0, n_pix=np.pi*5**2))

175.44534603043087


In [124]:
""" 
this test uses the default parameters set in second block, but the values aren't accurate due to 
the magnitude zero point being 24, once we have a better assesment for the mag_zp this should return better values
"""

# example run 
if __name__ == '__main__':
 
    #time to reach SNR=10 at two wavelengths
    print("time to reach SNR = 10:")
    for wave in [500, 550, 700]:
        print(f"\n  wavelength = {wave} nm  (QE = {get_qe(wave):.2f})")
        for m in [14, 16, 18, 20]:
            t = get_exptime(10, m, wave_nm=wave)
            print(f"    mag {m}  ->  {t:.0f} s  ({t/60:.1f} min)")
 
    

time to reach SNR = 10:

  wavelength = 500 nm  (QE = 0.89)
    mag 14  ->  0 s  (0.0 min)
    mag 16  ->  0 s  (0.0 min)
    mag 18  ->  1 s  (0.0 min)
    mag 20  ->  5 s  (0.1 min)

  wavelength = 550 nm  (QE = 0.84)
    mag 14  ->  0 s  (0.0 min)
    mag 16  ->  0 s  (0.0 min)
    mag 18  ->  1 s  (0.0 min)
    mag 20  ->  6 s  (0.1 min)

  wavelength = 700 nm  (QE = 0.46)
    mag 14  ->  0 s  (0.0 min)
    mag 16  ->  0 s  (0.0 min)
    mag 18  ->  1 s  (0.0 min)
    mag 20  ->  10 s  (0.2 min)
